# 15 - Pricing Config Schema

Every bursar deployment starts with a pricing configuration — a YAML file (or its equivalent Python dict) that defines how credits are charged for model inference, tool usage, search, cache, batch jobs, and subscription plans. This notebook walks through every field in that schema, from the simplest required fields to the most nested policy definitions.

The pricing config is validated at load time by the `PricingConfig` Pydantic model in `bursar.config`. Expression strings are parsed and checked against the known metric variables before any request is priced, so typos fail early rather than silently producing wrong costs.

We'll build the config incrementally — starting with a minimal valid config and adding sections one at a time — so each field's role and constraints are clear.

In [ ]:
from decimal import Decimal
from pprint import pprint
from bursar.config import PricingConfig, load_config_from_dict, ConfigError
from bursar.expr import evaluate_expression
from bursar.engine import PricingEngine
from bursar.metrics import UsageMetrics, ToolCall
from bursar.interface.models import PlanDefinition, BucketDefinition, OperationPolicy, FeatureLimit

## Top-level schema

The pricing config has thirteen top-level keys, one of which is required:

| Key | Type | Required | Default |
|-----|------|----------|--------|
| `version` | `Literal[1]` | No | `1` |
| `metering` | `object` | **Yes** | — |
| `metering.models` | `dict[str, str]` | **Yes** | — |
| `metering.tools` | `dict[str, str]` | No | `{"*": "calls * 0"}` |
| `metering.search` | `str` | No | `None` |
| `metering.cache_discount` | `str` | No | `None` |
| `metering.flat_jobs` | `dict[str, Decimal]` | No | `{}` |
| `ledger` | `object` | No | `{"min_balance": 0, "signup_grant": 50}` |
| `ledger.min_balance` | `Decimal` | No | `0` |
| `ledger.signup_grant` | `int` | No | `50` |
| `ledger.buckets` | `dict[str, BucketDefinition]` | No | `None` |
| `plans` | `dict[str, PlanDefinition]` | No | `None` |
| `billing` | `object` | No | `None` |
| `billing.currency` | `str` | No | `"USD"` |
| `billing.subscriptions` | `dict[str, BillingOffer]` | No | `{}` |
| `billing.topups` | `dict[str, BillingCreditTopup]` | No | `{}` |

Only `metering.models` is strictly required — you must define at least one model pricing expression. Every other key is optional and has sensible defaults.

### 1. `version` — schema version

Must be the literal integer `1`. Currently the only valid version. Will increment if the schema changes in a backwards-incompatible way.

In [ ]:
# A version other than 1 is rejected at load time.
try:
    load_config_from_dict({"version": 2, "metering": {"models": {"_default": "input_tokens * 1"}}})
except (ConfigError, Exception) as e:
    print(f"Rejected: {e}")

## 2. `models` — model pricing expressions

The only required section. A dict mapping model identifiers to expression strings. Each expression is evaluated with the `UsageMetrics` variables as inputs.

**Keys** are model IDs (e.g. `gpt-4o`, `claude-sonnet-4-6`, `_default`). The special key `_default` is a fallback: if a request's model name doesn't match any key and `_default` exists, that expression is used. If neither matches, the engine raises a `ValueError`.

**Values** are expression strings using these available variables:

| Variable | Type | Description |
|----------|------|-------------|
| `input_tokens` | `int` | Prompt token count |
| `output_tokens` | `int` | Completion token count |
| `cache_read_tokens` | `int` | Tokens read from LLM context cache |
| `cache_write_tokens` | `int` | Tokens written to context cache |
| `tool_calls` | `int` | Total tool invocations across all tools |
| `search_queries` | `int` | Number of search queries issued |
| `search_results` | `int` | Number of search results processed |
| `web_search_calls` | `int` | Web search API calls |
| `code_exec_calls` | `int` | Code execution sandbox invocations |

The expression language supports `+`, `-`, `*`, `/`, `//`, `%`, comparisons (`<`, `>`, `<=`, `>=`, `==`, `!=`), boolean operators (`and`, `or`, `not`), and these functions: `min`, `max`, `if`, `tier`, `clamp`, `percentile`, `ceil`, `floor`, `round`. Exponentiation (`**`) is blocked for safety. See [Notebook 02](02_expression_language.ipynb) for the full expression language reference.

In [ ]:
# Minimal valid config: just a _default model expression.
config = load_config_from_dict({
    "metering": {"models": {"_default": "input_tokens * 10 + output_tokens * 30"}},
})
print(f"Valid: version={config.version}, models={len(config.metering.models)}")
pprint(config.metering.models)

In [ ]:
# Full models section with multiple providers and a fallback.
full_models = {
    "metering": {
        "models": {
            "gpt-4o": "input_tokens * 5 + output_tokens * 15",
            "claude-sonnet-4-6": "input_tokens * 3 + output_tokens * 10",
            "claude-haiku-4-5": "input_tokens * 1 + output_tokens * 4",
            "_default": "input_tokens * 10 + output_tokens * 30",
        },
    },
}
config = load_config_from_dict(full_models)
print(f"{len(config.metering.models)} models registered")

In [ ]:
# Models section must be non-empty — this fails.
try:
    load_config_from_dict({"metering": {"models": {}}})
except ConfigError as e:
    print(f"Rejected: {e}")

### 3. `tools` — per-tool pricing

Defines additional credit costs per tool invocation. Keys are tool names (matching `ToolCall.name` in `UsageMetrics.tool_calls`). The special key `_default` covers any tool not explicitly listed.

Inside tool expressions, an additional variable is available:
- `this_tool_calls` — counts only calls matching this specific tool key (or, for `_default`, the count of unmatched calls). Use this instead of `tool_calls` for per-tool granularity.

In [ ]:
# Tools section with explicit keys and a wildcard fallback.
config = load_config_from_dict({
    "metering": {
        "models": {"_default": "input_tokens * 10 + output_tokens * 30"},
        "tools": {
            "code_exec": "calls * 50",       # 50 credits per code_exec call
            "web_fetch": "calls * 10",        # 10 credits per web_fetch call
            "*": "calls * 0",                  # everything else free
        },
    },
})

# When omitted, the default tools config is {"*": "calls * 0"}
print(f"Default tools wildcard key matches any tool not explicitly listed")
print(f"  tools expression uses 'calls' variable (not 'this_tool_calls' or 'tool_calls')")

### 4. `search` — search/RAG pricing

A single expression string for search-augmented generation costs. Uses the standard metric variables: `search_queries`, `search_results`, etc.

When set to `None` (or omitted entirely), the search cost component is zero.

In [ ]:
# Search expression: per-query + per-result, capped at 200.
config = load_config_from_dict({
    "metering": {
        "models": {"_default": "input_tokens * 10"},
        "search": "min(search_queries * 10 + search_results * 1, 200)",
    },
})
print(f"Search expression: {config.metering.search}")

# Verify with the engine:
engine = PricingEngine(config)
cost = engine.calculate(UsageMetrics(model="_default", input_tokens=100, search_queries=3, search_results=45))
print(f"Search cost: {cost.search_credits} (3*10 + 45*1 = 75, capped at 200)")


### 5. `cache` — cache discount

A single expression string that computes a cache discount (negative cost, reducing the total). Uses `cache_read_tokens` and `cache_write_tokens` among the standard variables.

The expression evaluates to a negative value (savings) or zero. The final total is clamped to `>= 0`.

In [ ]:
# Cache discount expression: -cache_read_tokens * 0.5 (50% of read cost back), capped at model cost.
config = load_config_from_dict({
    "metering": {
        "models": {"_default": "input_tokens * 10 + output_tokens * 30"},
        "cache_discount": "clamp(-cache_read_tokens * 0.5, -(input_tokens * 10 + output_tokens * 30), 0)",
    },
})
print(f"Cache expression: {config.metering.cache_discount}")


### 6. `min_balance` — minimum balance floor

A `Decimal` value (non-negative) representing the minimum credit balance a user must maintain. Enforced in `strict` billing mode. Prevents negative balances from concurrent reservation races. Defaults to `0`.

In [ ]:
config = load_config_from_dict({
    "metering": {"models": {"_default": "input_tokens * 10"}},
    "ledger": {"min_balance": 5000},
})
print(f"Min balance: {config.ledger.min_balance}")



### 7. `signup_bonus` — new-user credit grant

An integer (>= 0) specifying how many credits a new user receives on signup. Defaults to `50`. Used by `CreditManager` when creating new user records.

In [ ]:
config = load_config_from_dict({
    "metering": {"models": {"_default": "input_tokens * 10"}},
    "ledger": {"signup_grant": 100},
})
print(f"Signup grant: {config.ledger.signup_grant}")



### 8. `fixed` — flat-rate batch jobs

Defines fixed credit charges for batch or async jobs that don't scale with token counts. Keys are job names, values are `Decimal` credit amounts (must be >= 0).

Fixed costs bypass the model/tool/search/cache calculation entirely. The engine looks up the cost by `UsageMetrics.fixed_job`. If the job name isn't in `fixed`, the engine returns 0 for the fixed component.

Common use cases: roadmap generation, review/quiz sessions, diagnostic assessments.

In [ ]:
config = load_config_from_dict({
    "metering": {
        "models": {"_default": "input_tokens * 10"},
        "flat_jobs": {
            "roadmap_gen": 5000,
            "review": 2000,
            "diagnostic": 1500,
        },
    },
})
print(f"Flat jobs: {dict(config.metering.flat_jobs)}")

# Loading a config with negative fixed cost is rejected.
try:
    load_config_from_dict({
        "metering": {
            "models": {"_default": "input_tokens * 10"},
            "flat_jobs": {"negative_job": -100},
        },
    })
except Exception as e:
    print(f"Rejected negative fixed cost: {e}")


---

## Subscription offers (`subscriptions`)

The `subscriptions` key defines the available subscription offers that users can purchase through a payment provider (Stripe, Dodo). Each offer maps a payment-provider product/price to a bursar plan and entitlement mode.

Entries are validated against `BillingOffer` at config load time:

In [ ]:
from bursar.billing.models import BillingOffer, BillingCreditTopup, BillingConfig

print("BillingOffer fields:")
for name, field in BillingOffer.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

### BillingOffer fields

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `offer_key` | `str` | — | Unique offer identifier (the dict key itself) |
| `plan_key` | `str` | — | References a plan key in the `plans` section |
| `interval` | `"day"` / `"week"` / `"month"` / `"year"` | `"month"` | Billing interval for the subscription |
| `interval_count` | `int` | `1` | Number of intervals per billing period (e.g. 3 months) |
| `entitlement_mode` | `"allowance"` / `"cycle_grant"` | `"allowance"` | How the user receives credits each cycle |
| `cycle_grant_credits` | `int \| None` | `None` | Fixed credits granted each cycle (only in `cycle_grant` mode) |
| `cycle_grant_tier` | `str \| None` | `None` | Tier to grant credits into (default: `"purchased"`) |
| `cycle_grant_replace_prior` | `bool` | `True` | Whether to revoke prior cycle grants before granting new ones |
| `provider_refs` | `dict[str, BillingSubscriptionOfferRef]` | `{}` | Mapping of provider → product/price IDs |

### entitlement_mode

Controls what happens when a subscription billing cycle completes:

- **`"allowance"`** (default) — The user's plan free allowance resets (the `allowance_period` on the referenced plan controls the reset window). The user gets their plan's `free_allowance` credits back to spend.
- **`"cycle_grant"`** — A fixed number of credits (`cycle_grant_credits`) is deposited into the user's balance each cycle, deposited into the specified `cycle_grant_tier`. When `cycle_grant_replace_prior` is `True`, any previous cycle grants are revoked first (so the user gets fresh credits each period rather than accumulating them).

### Interval examples

| `interval` | `interval_count` | Effective billing period |
|------------|-------------------|-------------------------|
| `"month"` | `1` | Monthly (default) |
| `"month"` | `3` | Quarterly |
| `"year"` | `1` | Annual |
| `"day"` | `7` | Weekly |
| `"week"` | `2` | Bi-weekly |

In [ ]:
# Subscriptions section with allowance-mode and cycle-grant offers.
config = load_config_from_dict({
    "metering": {"models": {"_default": "input_tokens * 10"}},
    "plans": {
        "seeker": {
            "label": "Seeker", "allowance": {"amount": 5000, "period": "calendar_month"},
        },
        "sage": {
            "label": "Sage", "allowance": {"amount": 50000, "period": "calendar_month"},
        },
    },
    "billing": {
        "subscriptions": {
            "seeker-monthly": {
                "plan": "seeker",
                "interval": "month",
                "interval_count": 1,
                "grant": {"mode": "allowance"},
                "providers": {
                    "stripe": {
                        "price_id": "price_monthly_seeker",
                    },
                },
            },
            "sage-annual": {
                "plan": "sage",
                "interval": "year",
                "interval_count": 1,
                "grant": {"mode": "cycle_grant", "credits": 50000, "bucket": "purchased", "replace_prior": True},
                "providers": {
                    "stripe": {
                        "price_id": "price_annual_sage",
                    },
                },
            },
        },
    },
})
print(f"{len(config.billing.subscriptions)} subscription offers configured")
for key, offer in config.billing.subscriptions.items():
    refs = {p: r.price_id for p, r in (offer.providers or {}).items()}
    print(f"  {key}: plan={offer.plan}, interval={offer.interval}, mode={offer.grant.mode}, refs={refs}")


### BillingSubscriptionOfferRef

Each entry in `provider_refs` maps a payment provider to a specific product/price that the provider uses for billing:

| Field | Type | Description |
|-------|------|-------------|
| `provider` | `str` | Payment provider name (`"stripe"`, `"dodo"`) |
| `product_id` | `str \| None` | Provider product ID |
| `price_id` | `str \| None` | Provider price ID |
| `variant_id` | `str \| None` | Provider variant ID (Dodo-specific) |
| `lookup_key` | `str \| None` | Lookup key (used as a fallback when no product/price match) |

The `BillingStore` resolves incoming payment events against these refs by matching on `product_id` → `price_id` (in that priority order). If neither matches but a `lookup_key` is present on the event, the `lookup_key` is used as the offer key directly.

**Supported providers**: `"stripe"` and `"dodo"` (enum `BillingProvider`).

---

## Credit top-ups (`credit_topups`)

The `credit_topups` key defines credit-pack products that users can purchase à la carte through a payment provider. Each topup maps a provider product/price to a credit amount and tier.

In [ ]:
print("BillingCreditTopup fields:")
for name, field in BillingCreditTopup.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `tier` | `str` | `"purchased"` | Credit tier for deposited credits |
| `currency` | `str` | `"USD"` | Currency code for price validation |
| `credits_per_major_unit` | `int` | `1000` | Credits per 1 major unit (e.g. 1000 credits per $1) |
| `min_amount_minor` | `int` | `500` | Minimum purchase in minor units (e.g. 500 = $5.00) |
| `max_amount_minor` | `int` | `500000` | Maximum purchase in minor units (e.g. 500000 = $5000.00) |
| `tax_behavior` | `"exclude_tax"` / `"include_tax"` | `"exclude_tax"` | How tax is handled in provider pricing |
| `provider_refs` | `dict[str, BillingSubscriptionOfferRef]` | `{}` | Mapping of provider → product/price IDs |

When a payment succeeds for a credit topup product, `BillingManager` validates the amount against `min_amount_minor` / `max_amount_minor`, computes the credits using `credits_per_major_unit`, and deposits them into the user's balance via `CreditManager.add_credits()`. If a refund occurs, those credits are clawed back.

In [ ]:
# Credit topups section with multiple packs.
config = load_config_from_dict({
    "metering": {"models": {"_default": "input_tokens * 10"}},
    "billing": {
        "topups": {
            "small-pack": {
                "deposit_to": "purchased",
                "credits_per_unit": 1000,
                "min_amount_minor": 500,
                "max_amount_minor": 50000,
                "providers": {
                    "stripe": {
                        "price_id": "price_credits_small",
                    },
                },
            },
            "large-pack": {
                "deposit_to": "purchased",
                "credits_per_unit": 1200,
                "min_amount_minor": 10000,
                "max_amount_minor": 500000,
                "providers": {
                    "stripe": {
                        "price_id": "price_credits_large",
                    },
                },
            },
        },
    },
})
print(f"{len(config.billing.topups)} credit topup products configured")
for key, topup in config.billing.topups.items():
    print(f"  {key}: tier={topup.deposit_to}, credits_per_dollar={topup.credits_per_unit}, min={topup.min_amount_minor}, max={topup.max_amount_minor}")


---

## BillingConfig — wiring subscriptions and topups together

The `subscriptions` and `credit_topups` sections are loaded as a `BillingConfig` instance that `BillingManager` syncs to the billing store at startup:

In [ ]:
print("BillingConfig fields:")
for name, field in BillingConfig.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

# Build a BillingConfig from the pricing config's subscription/topup sections.
config = load_config_from_dict({
    "metering": {"models": {"_default": "input_tokens * 10"}},
    "plans": {"seeker": {"label": "Seeker"}},
    "billing": {
        "subscriptions": {
            "pro-monthly": {
                "plan": "seeker",
                "interval": "month",
                "providers": {
                    "stripe": {"price_id": "price_pro_monthly"},
                },
            },
        },
        "topups": {
            "credits-10": {
                "deposit_to": "purchased",
                "credits_per_unit": 1000,
                "providers": {
                    "stripe": {"price_id": "price_credits_10"},
                },
            },
        },
    },
})

# Build a BillingConfig from the pricing config's billing section.
bc = BillingConfig(
    subscriptions={
        k: v
        for k, v in (config.billing.subscriptions or {}).items()
    },
    topups={
        k: v
        for k, v in (config.billing.topups or {}).items()
    },
)
print(f"BillingConfig has {len(bc.subscriptions)} subscriptions and {len(bc.topups)} topups")
for key, offer in bc.subscriptions.items():
    print(f"  Sub: {key} → plan={offer.plan}, interval={offer.interval}")
for key, topup in bc.topups.items():
    print(f"  Topup: {key} → tier={topup.deposit_to}, rate={topup.credits_per_unit}/$")




## BillingManager and PaymentProvider integration

The `BillingManager` (`bursar.billing.manager`) is the bridge between payment-provider webhook events and the `CreditManager`. It is not part of the YAML config directly, but it consumes the `BillingConfig` that is derived from the `subscriptions` / `credit_topups` sections:

```
PaymentProvider (Stripe/Dodo)
  └─ webhook → BillingManager.handle_event(event)
                   ├─ claim_billing_event()  → idempotency
                   ├─ route to handler       (customer.created, subscription.activated,
                   │                          invoice.paid, payment.succeeded, refund.created, ...)
                   ├─ upsert_billing_subscription()
                   └─ CreditManager.set_user_plan() / add_credits()
```

**Built-in providers**:

- `StripeProvider(bm, webhook_secret)` — Stripe checkout, portal, webhooks
- `DodoProvider(get_client, config, bm)` — Dodo Payments checkout, portal, webhooks
- `MockPaymentProvider` — In-memory mock for testing

**PaymentProvider interface** (`bursar.providers.types`):

In [ ]:
# Minimal BillingManager wiring example with a MockPaymentProvider.
from bursar.billing import BillingManager, MemoryBillingStore, BillingConfig
from bursar.manager import CreditManager
from bursar.interface.memory import MemoryStore

# Step 1: Create the BillingConfig from the pricing config.
config = load_config_from_dict({
    "metering": {
        "models": {"_default": "input_tokens * 10"},
    },
    "plans": {
        "pro": {
            "label": "Pro", "allowance": {"amount": 10000, "period": "calendar_month"},
        },
    },
    "billing": {
        "subscriptions": {
            "pro-monthly": {
                "plan": "pro",
                "interval": "month",
                "grant": {"mode": "allowance"},
                "providers": {
                    "mock": {"price_id": "price_pro_monthly"},
                },
            },
        },
    },
})

billing_config = BillingConfig(
    subscriptions={
        k: v
        for k, v in (config.billing.subscriptions or {}).items()
    },
)

# Step 2: Wire up stores and managers.
billing_store = MemoryBillingStore()
credit_mgr = CreditManager(store=MemoryStore())
bm = BillingManager(
    billing_store=billing_store,
    credit_manager=credit_mgr,
    config=billing_config,
)

# Step 3: Create a MockPaymentProvider pointing at the BillingManager.
from bursar.providers.mock.provider import MockPaymentProvider
provider = MockPaymentProvider(bm)

print("Billing pipeline ready:")
print(f"  Store has {len(billing_store._offers)} offers synced")
print(f"  Provider: {provider.provider}")
print(f"  User plans: {credit_mgr.get_user_plan('user_123')}")


### Billing event lifecycle

When a provider webhook fires, `BillingManager.handle_event()` processes it:

1. **Claim** — Records the event as claimed (idempotency via `claim_billing_event()`)
2. **Route** — Dispatches to the correct handler by `event_type`
3. **Resolve user** — Maps the provider customer ID to a bursar user ID via billing store + optional `resolve_user` callback
4. **Sync subscription** — Creates/updates `BillingSubscriptionState` in the store
5. **Provision** — Calls `CreditManager.set_user_plan()` and optionally grants cycle credits (`cycle_grant` mode)
6. **Complete** — Marks the event as completed

**Supported event types** (`BillingEventType`):

| Category | Events |
|----------|--------|
| Customer | `customer.created`, `customer.updated`, `customer.deleted` |
| Checkout | `checkout.completed`, `checkout.expired` |
| Subscription | `subscription.created`, `.updated`, `.activated`, `.renewed`, `.plan_changed`, `.cancellation_scheduled`, `.cancellation_unscheduled`, `.canceled`, `.expired`, `.paused`, `.resumed`, `.trial_will_end` |
| Invoice | `invoice.created`, `.finalized`, `.finalization_failed`, `.upcoming`, `.paid`, `.payment_failed`, `.payment_action_required`, `.voided` |
| Payment | `payment.succeeded`, `.failed` |
| Refund | `refund.created`, `.updated`, `.failed` |
| Dispute | `dispute.created`, `.closed` |
| Payment method | `payment_method.attached`, `.updated`, `.detached` |

A `resolve_user` callback can be provided to link provider customer IDs to internal user IDs when the customer doesn't exist yet in the billing store.

In [ ]:
# Minimal BillingManager wiring example with a MockPaymentProvider.
from bursar.billing import BillingManager, MemoryBillingStore, BillingConfig
from bursar.manager import CreditManager
from bursar.interface.memory import MemoryStore

# Step 1: Create the BillingConfig from the pricing config.
config = load_config_from_dict({
    "metering": {
        "models": {"_default": "input_tokens * 10"},
    },
    "plans": {
        "pro": {
            "label": "Pro", "allowance": {"amount": 10000, "period": "calendar_month"},
        },
    },
    "billing": {
        "subscriptions": {
            "pro-monthly": {
                "plan": "pro",
                "interval": "month",
                "grant": {"mode": "allowance"},
                "providers": {
                    "mock": {"price_id": "price_pro_monthly"},
                },
            },
        },
    },
})

billing_config = BillingConfig(
    subscriptions={
        k: v
        for k, v in (config.billing.subscriptions or {}).items()
    },
)

# Step 2: Wire up stores and managers.
billing_store = MemoryBillingStore()
credit_mgr = CreditManager(store=MemoryStore())
bm = BillingManager(
    billing_store=billing_store,
    credit_manager=credit_mgr,
    config=billing_config,
)

# Step 3: Create a MockPaymentProvider pointing at the BillingManager.
from bursar.providers.mock.provider import MockPaymentProvider
provider = MockPaymentProvider(bm)

print("Billing pipeline ready:")
print(f"  Store has {len(billing_store._offers)} offers synced")
print(f"  Provider: {provider.provider}")
print(f"  User plans: {credit_mgr.get_user_plan('user_123')}")


---

## Credit tiers (`tiers`)

Credit tiers control the order in which credit buckets are consumed, whether they expire, and overdraft behavior. Each tier is a bucket of credits with a priority. Higher priority (larger number = lower priority) buckets are consumed first.

Field schema for each tier entry:

In [ ]:
print(BucketDefinition.model_json_schema())
print()
print("Fields:")
for name, field in BucketDefinition.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")


**Tier priority walk**: when deducting credits, bursar walks tiers from lowest priority number (highest priority) to highest priority number. Gifted credits (priority 10) are consumed first, then monthly allowance (priority 20), then purchased credits (priority 30).

**Constraints enforced at config load time:**
- At most one tier may set `allow_overdraft=True`
- At most one tier may set `is_default=True` (new credits without an explicit tier key land here)
- `default_ttl_days` > 0 when set
- An explicit `tiers: {}` is rejected (omit the key entirely for "no tiers")

In [ ]:
# Full buckets section — the standard three-tier setup.
config = load_config_from_dict({
    "metering": {"models": {"_default": "input_tokens * 10"}},
    "ledger": {
        "buckets": {
            "gifted": {
                "label": "Gifted Credits",
                "priority": 10,
                "expires": True,
                "ttl_days": 7,
            },
            "monthly": {
                "label": "Monthly Allowance",
                "priority": 20,
                "expires": True,
                "ttl_days": 30,
            },
            "purchased": {
                "label": "Purchased Credits",
                "priority": 30,
                "expires": False,
                "allow_overdraft": True,
                "default": True,
            },
        },
    },
})
print(f"{len(config.ledger.buckets)} buckets configured")
for k, t in config.ledger.buckets.items():
    print(f"  {k}: priority={t.priority}, expires={t.expires}, overdraft={t.allow_overdraft}, default={t.default}")


---

## Subscription plans (`plans`)

Plans define subscription tiers with free monthly allowances, rate overrides, feature flags, and per-operation financial-safety policies. Users without a plan use the constructor defaults from `CreditManager`.

In [ ]:
print("PlanDefinition fields:")
for name, field in PlanDefinition.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

### Plan field details

| Field | Type | Description |
|-------|------|-------------|
| `id` | `str` | Unique plan identifier (required) |
| `name` | `str` | Human-readable plan name (required) |
| `free_allowance` | `Decimal` | Free credits per billing period, default 0 |
| `default_billing_mode` | `"strict"` / `"overdraft"` | Plan-wide billing mode, default `"strict"` |
| `rate_overrides` | `dict[str, str] \| None` | Per-model expression overrides (optional) |
| `features` | `dict[str, Any] \| None` | Arbitrary feature flags for entitlement checks |
| `feature_limits` | `dict[str, FeatureLimit] \| None` | Per-feature invocation-count limits |
| `per_operation` | `dict[str, OperationPolicy] \| None` | Per-operation-type policy overrides |
| `max_concurrent` | `int \| None` | Plan-wide cap on simultaneously-active billed operations |
| `overdraft_floor` | `Decimal \| None` | Negative balance floor in overdraft mode |
| `allowance_period` | `"calendar_month"` / `"rolling_30d"` / `"anniversary"` | Free-allowance reset window |

**Accepted alias**: `billing_mode` is accepted as a short-form alias for `default_billing_mode` in YAML.

### allowance_period modes

Controls when a user's free allowance resets (see `bursar.allowance` for the full resolver):

| Mode | Reset trigger | Anchor required? | Use case |
|------|---------------|------------------|----------|
| `"calendar_month"` | 1st of each UTC month | No | Simple monthly billing (default) |
| `"rolling_30d"` | Every 30 days from plan assignment | Yes (`plan_assigned_at`) | Fair 30-day windows regardless of signup date |
| `"anniversary"` | Monthly on plan-assignment day-of-month, clamped to month length | Yes (`plan_assigned_at`) | Anniversary-based plans |

When `"rolling_30d"` or `"anniversary"` is configured but no anchor is available (user never assigned a plan), both fall back to `"calendar_month"` — no crash, just non-customized windows.

In [ ]:
# Feature limit example: cap roadmap_gen to 1x per day, review to 10x per month.
config = load_config_from_dict({
    "metering": {"models": {"_default": "input_tokens * 10"}},
    "plans": {
        "seeker": {
            "label": "Seeker",
            "allowance": {"amount": 5000, "period": "calendar_month"},
            "entitlements": {
                "roadmap_gen": {
                    "max_calls": 1,
                    "period": "daily",
                    "on_exceed": "deny",
                },
                "review": {
                    "max_calls": 10,
                    "period": "monthly",
                    "on_exceed": "warn",
                },
            },
        },
    },
})
seeker = config.plans["seeker"]
print(f"Plan: {seeker.label}")
if seeker.entitlements:
    for feat_name, limit in seeker.entitlements.items():
        print(f"  {feat_name}: max {limit.max_calls}x per {limit.period}, action={limit.on_exceed}")


### OperationPolicy

Per-operation policy overrides for financial safety:

In [ ]:
print("OperationPolicy fields:")
for name, field in OperationPolicy.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

| Field | Type | Description |
|-------|------|-------------|
| `billing_mode` | `"strict"` / `"overdraft"` | Per-operation billing mode override |
| `max_concurrent` | `int \| None` | Max simultaneous leases for this operation type |
| `overdraft_floor` | `Decimal \| None` | Per-operation overdraft floor |

**Policy resolution order** (most specific wins):
1. Per-call `billing_mode` argument to `reserve()` / `run_billed()`
2. `plan.per_operation[operation_type]`
3. `plan.default_billing_mode`
4. Constructor preset on `CreditManager`

### FeatureLimit

Per-feature invocation-count limits for cadence-based rate limiting:

In [ ]:
print("FeatureLimit fields:")
for name, field in FeatureLimit.model_fields.items():
    print(f"  {name}: {field.annotation}  (default: {field.default})")

| Field | Type | Description |
|-------|------|-------------|
| `max_calls` | `int` | Maximum invocations within one period |
| `period` | `"daily"` / `"weekly"` / `"monthly"` / `"yearly"` | Calendar-aligned window |
| `action` | `"deny"` / `"warn"` / `"notify"` | Action when limit reached |

## How the engine combines cost components

The `PricingEngine.calculate()` method evaluates **five independent cost components** and sums them. Each component comes from a separate config section:

| Component | Config source | Expression variables | Sign |
|-----------|---------------|---------------------|------|
| `model_credits` | `models[model_name]` | Standard metrics (input_tokens, output_tokens, etc.) | Positive (cost) |
| `tool_credits` | `tools[tool_name]` | Standard + `this_tool_calls` | Positive (cost) |
| `search_credits` | `search` (single expression) | Standard | Positive (cost) |
| `cache_savings` | `cache` (single expression) | Standard | Negative (discount) |
| `fixed_credits` | `fixed[job_name]` (lookup, not expression) | N/A — flat Decimal | Positive (cost) |

The engine then produces a `CostBreakdown`:

```
raw_total = model_credits + tool_credits + search_credits + cache_savings + fixed_credits
total = max(0, raw_total)   # never negative
```

Each field is quantized to 4 decimal places with `ROUND_HALF_UP`. The `CostBreakdown.total` is the single source of truth — computed once, not re-derived.

When a config section is omitted (e.g. no `search` key), its component is zero. When a model name doesn't match any key, the engine falls back to `_default` or raises `ValueError`.

The `subscriptions` and `credit_topups` sections are **not** part of the pricing engine — they are consumed separately by `BillingManager` to handle payment-provider webhook events and provision credits.

## Complete config example

The following config demonstrates every supported field together — all sections, all sub-fields, all nested structures in one dict.

In [ ]:
config = load_config_from_dict({
    "metering": {"models": {"_default": "input_tokens * 10"}},
    "ledger": {"min_balance": 5000},
})
print(f"Min balance: {config.ledger.min_balance}")



## Putting it together: the complete YAML

Every field documented above in one YAML file:

```yaml
version: 1

models:
  claude-sonnet-4-6: input_tokens * 15 + output_tokens * 45
  claude-haiku-4-5: input_tokens * 3 + output_tokens * 9
  _default: input_tokens * 10 + output_tokens * 30

tools:
  code_exec: this_tool_calls * 50
  web_fetch: this_tool_calls * 10
  _default: tool_calls * 0

search: clamp(min(search_queries * 5 + search_results * 1, 100), 0, 500)

cache: clamp(-cache_read_tokens * 0.002, -(input_tokens * 10 + output_tokens * 30), 0)

min_balance: 5000
signup_bonus: 50

fixed:
  roadmap_gen: 20
  diagnostic: 15
  review: 5

tiers:
  gifted:
    name: Gifted Credits
    priority: 10
    expires: true
    default_ttl_days: 365
  monthly:
    name: Monthly Allowance
    priority: 20
    expires: true
    default_ttl_days: 30
  purchased:
    name: Purchased Credits
    priority: 30
    expires: false
    allow_overdraft: true
    is_default: true

plans:
  seeker:
    id: seeker
    name: Seeker
    free_allowance: 5000
    default_billing_mode: strict
    max_concurrent: 1
    overdraft_floor: 0
    allowance_period: calendar_month
    rate_overrides:
      claude-sonnet-4-6: min(input_tokens * 3 + output_tokens * 9, 50)
    per_operation:
      review:
        billing_mode: strict
        max_concurrent: 1
        overdraft_floor: 0
      roadmap_gen:
        billing_mode: strict
        max_concurrent: 1
        overdraft_floor: 0
    feature_limits:
      roadmap_gen:
        max_calls: 1
        period: daily
        action: deny
    features:
      max_daily_roadmaps: 1
      max_concurrency: 1

  sage:
    id: sage
    name: Sage
    free_allowance: 50000
    default_billing_mode: strict
    max_concurrent: 3
    overdraft_floor: 0
    allowance_period: calendar_month
    per_operation:
      review:
        billing_mode: strict
        max_concurrent: 3
        overdraft_floor: 0
    feature_limits:
      roadmap_gen:
        max_calls: 5
        period: daily
        action: deny
    features:
      max_daily_roadmaps: 5
      max_concurrency: 3

subscriptions:
  seeker-monthly:
    plan_key: seeker
    interval: month
    interval_count: 1
    entitlement_mode: allowance
    provider_refs:
      stripe:
        provider: stripe
        price_id: price_1QxExampleSeekerMonthly
      dodo:
        provider: dodo
        product_id: prod_dodo_seeker_monthly

  sage-annual:
    plan_key: sage
    interval: year
    interval_count: 1
    entitlement_mode: cycle_grant
    cycle_grant_credits: 50000
    cycle_grant_tier: purchased
    cycle_grant_replace_prior: true
    provider_refs:
      stripe:
        provider: stripe
        price_id: price_1QxExampleSageAnnual

credit_topups:
  small-pack:
    tier: purchased
    currency: USD
    credits_per_major_unit: 1000
    min_amount_minor: 500
    max_amount_minor: 50000
    provider_refs:
      stripe:
        provider: stripe
        price_id: price_1QxExampleSmallPack

  large-pack:
    tier: purchased
    currency: USD
    credits_per_major_unit: 1200
    min_amount_minor: 10000
    max_amount_minor: 500000
    tax_behavior: exclude_tax
    provider_refs:
      stripe:
        provider: stripe
        product_id: prod_large_credits
```

In [ ]:
# For reference, the full schema of PricingConfig as JSON Schema:
import json
schema = PricingConfig.model_json_schema()
print(json.dumps(schema, indent=2))